# CNN — ch3_cifar10

Demo CPU của **3 checkpoint đã train riêng**: scratch/Keras/PyTorch.
Không train hoặc gọi TEST. Bộ dự đoán serving thực tế là NumPy portable, không phải ba native GPU runtime.

Lý thuyết, mẫu/distribution và metric: `report/sections/v3/ch03_cnn.md`.
Hàm CNN: `src/common/numpy_engine.py`, `src/common/backend_models.py`; trainer: `src/ch03_cnn/training.py`.
Dữ liệu y khoa chỉ dùng học thuật, không chẩn đoán.

In [ ]:
from pathlib import Path
import os, sys
root = Path.cwd().resolve()
while not (root / 'TIEU_LUAN_AI_V3_EXECUTION_PLAN.md').exists():
    if root.parent == root:
        raise RuntimeError('Open notebook inside the repository')
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from scripts.demo_phase27 import infer, sample_request
from src.common.contracts import read_json
from src.ch03_cnn.protocol import BASE
import pandas as pd

In [ ]:
request, train_id = sample_request('ch3_cifar10')
job = infer(request, seed=42)
print('Fixed TRAIN sample:', train_id)
pd.DataFrame([{'training_implementation': r['implementation'], 'serving_backend': r['actual_backend'], 'device': r['actual_device'], 'label': r['prediction']['label'], 'inference_ms_single_call': r['inference_ms']} for r in job['results']])

Một lời gọi demo không thay thế benchmark 5 warmup/30 repeats. Bảng quality/efficiency dưới đây chỉ đọc metric đã lưu; không đánh giá lại TEST.

In [ ]:
release = read_json(BASE / 'release.json')
rows = []
for e in release['runs']:
    if e['dataset_id'] != 'ch3_cifar10':
        continue
    path = root / 'artifacts/metrics/phase27/test' / (e['run_id'] + '.json')
    if not path.exists():
        continue
    record = read_json(path)
    values = {m['name']: m['value'] for m in record['metrics']}
    rows.append({'method': e['method'], 'seed': e['seed'], 'macro_f1': values['macro_f1'], 'accuracy': values['accuracy'], 'train_seconds': record['efficiency']['train_seconds']})
pd.DataFrame(rows)

In [ ]:
assert not any(name in sys.modules for name in ('torch', 'tensorflow', 'keras'))
assert job['comparison_valid']
print('CPU/API-contract PASS. TEST was not executed.')